# 56 Veeva Link candidate features on the spine (Phase 0, U6 + U7)

Builds the candidate feature set of plan `50_` section 5.1 for **all 294,727 HCPs** on the spine (`54_`), from the frozen snapshot (`52_`), the config (`53_`) and the long activity tables (`55_`). This replaces the 100-`SE_ID` pilot of `11_` (U6):

* keyed on the spine, not `in_coh()`;
* disease and class deciles from `CLINICAL_LEADER` / `FOCUS_AREA` / `DIGITAL_REACH`, never from the deprecated `COMMUNITY_LEADER_FOCUS_AREA`;
* composites standardised on the full matched universe;
* `DIGITAL_REACH` joins on string `SE_ID` (the "CG"-prefix mismatch of the pilot does not occur; 4,710 HCPs scored in `54_`).

**As-of rule (U7).** For anchor month `A`, T1 windows end at `E = min(A, table data cut-off)` from the `52_` manifest. Trials and publications use 5 years, events and payments 3 years (`config/windows_v1`). Payment program year `Y` counts only if it was published by `E` (assumed 30 June of `Y + 1`).

**Fixed schema.** Column names and order come from `FEATURE_SPEC` (written to `config/feature_schema_v1`); output must match it on every run.

**Missing is not zero.** `NOT_IN_VEEVA` HCPs get null on every feature except the coverage flags; in-Veeva HCPs with no activity get 0.

**Outputs:** `features/t1_t2/anchor=<A>/` (model candidates) and `features/t3/anchor=<A>/` (vendor scores, read only by experiments E5 and E6, decision D2).

In [0]:
from pyspark.sql import functions as F
import pandas as pd, datetime as dt

OUT = "/Volumes/ph-com-ai-us-prod-catalog-2553213839599380/kerendia_nba_3/kerendia_nba_3_1_volume/hcp_level_model/dev_pipeline/notebook_output/veeva_link_to_people/"
SNAPSHOT_ID = "load=2025-06-30"
SNAP, CFG, ACT, FEAT = f"{OUT}snapshot/{SNAPSHOT_ID}/", f"{OUT}config/", f"{OUT}activity/", f"{OUT}features/"
rd = lambda t: spark.read.parquet(f"{SNAP}{t}")

spine = spark.read.parquet(f"{OUT}spine/veeva_npi_se_map")
matched = spine.filter("IN_VEEVA = 1").select("SE_ID").distinct()
CUTOFF = {r.table: dt.date.fromisoformat(r.data_cutoff) for r in spark.read.parquet(f"{SNAP}_manifest").collect()}
WIN = {r.families: r.months for r in spark.read.parquet(CFG + "windows_v1").collect()}
M_TRIAL_PUB, M_EVENT_PAY = WIN["trials; publications"], WIN["events; payments"]
COMPETITORS = ["astrazeneca", "boehringer", "lilly", "janssen", "novo nordisk"]   # SGLT2 / GLP-1 / MRA competitors

def window(anchor, table, months):
    end = min(anchor, CUTOFF[table])
    start = (pd.Timestamp(end) - pd.DateOffset(months=months)).date()
    return start, end

ANCHORS = [dt.date(2025, 10, 23), dt.date(2024, 6, 30)]   # inference-like anchor and a pre-cut-off training-like anchor
for a in ANCHORS:
    print(a, {t: window(a, t, M_TRIAL_PUB if t in ("clinical_trial", "publication") else M_EVENT_PAY)
              for t in ["clinical_trial", "publication", "medical_event", "payment"]})

In [0]:
# (name, tier, family, rule). Order here = column order of the output
FEATURE_SPEC = [
    ("veeva_coverage_status", "T2", "coverage", "NOT_IN_VEEVA / IN_VEEVA_UNSCORED / SCORED"),
    ("veeva_has_profile", "T2", "coverage", "1 if matched to SCIENTIFIC_EXPERT on NPI"),
    ("veeva_has_inscope_decile", "T2", "coverage", "1 if any in-scope decile (dictionary v1)"),
    ("veeva_n_inscope_themes", "T2", "coverage", "distinct dictionary themes with a decile"),
    ("veeva_trial_any_5y", "T1", "trials", "any trial active in window"),
    ("veeva_trial_log_n_5y", "T1", "trials", "log1p distinct trials in window"),
    ("veeva_trial_pi_flag_5y", "T1", "trials", "PI on any trial in window"),
    ("veeva_trial_n_inscope_5y", "T1", "trials", "distinct in-scope trials in window"),
    ("veeva_trial_max_phase_inscope_5y", "T1", "trials", "max phase 1-4 of in-scope trials, 0 if none"),
    ("veeva_trial_active_flag", "T1", "trials", "trial running at window end"),
    ("veeva_trial_finerenone_flag_5y", "T1", "trials", "trial name tagged finerenone"),
    ("veeva_trial_mra_flag_5y", "T1", "trials", "trial name tagged MRA or finerenone"),
    ("veeva_trial_sglt2_flag_5y", "T1", "trials", "trial name tagged SGLT2"),
    ("veeva_trial_cardiorenal_n_5y", "T1", "trials", "distinct trials tagged CKD / cardiorenal / landmark / finerenone / MRA / SGLT2 / ERA"),
    ("veeva_pub_any_5y", "T1", "publications", "any publication in window"),
    ("veeva_pub_log_n_5y", "T1", "publications", "log1p publications in window"),
    ("veeva_pub_senior_author_flag_5y", "T1", "publications", "first or last author in window"),
    ("veeva_pub_n_inscope_5y", "T1", "publications", "in-scope publications in window"),
    ("veeva_pub_months_since_last", "T1", "publications", "months from window end to last publication, null if none ever"),
    ("veeva_event_any_3y", "T1", "events", "any event in window"),
    ("veeva_event_podium_flag_3y", "T1", "events", "speaker / chair / panelist / moderator in window"),
    ("veeva_event_poster_flag_3y", "T1", "events", "poster presenter in window"),
    ("veeva_event_n_inscope_3y", "T1", "events", "distinct in-scope events in window"),
    ("veeva_pay_any_3y", "T1", "payments", "any payment, published program years in window"),
    ("veeva_pay_log_total_3y", "T1", "payments", "log1p total USD"),
    ("veeva_pay_excl_food_log_3y", "T1", "payments", "log1p total USD excluding food and beverage"),
    ("veeva_pay_n_companies_3y", "T1", "payments", "distinct paying companies"),
    ("veeva_pay_speaker_consult_flag_3y", "T1", "payments", "speaker / faculty / consulting payment"),
    ("veeva_pay_competitor_flag_3y", "T1", "payments", "payment from a competitor company (not Bayer)"),
    ("veeva_pay_research_flag_3y", "T1", "payments", "research payment"),
    ("veeva_pay_bayer_flag_3y", "T1", "payments", "Bayer payment (separate flag only, decision D3)"),
    ("veeva_expert_type", "T2", "profile", "EXPERT_TYPE"),
    ("veeva_is_md_do", "T2", "profile", "MD or DO in DEGREE_1..3"),
    ("veeva_is_phd", "T2", "profile", "PhD in DEGREE_1..3"),
    ("veeva_is_academic", "T2", "profile", "affiliation looks academic (university / college / school of medicine)"),
    ("veeva_emerging_expert", "T2", "profile", "EMERGING_EXPERT = Y"),
    ("veeva_n_editorial_boards", "T2", "profile", "editorial board roles"),
    ("veeva_n_society_roles", "T2", "profile", "scientific / professional association roles"),
    ("veeva_kidney_society_flag", "T2", "profile", "association name mentions kidney / nephrology"),
    ("veeva_n_sponsors", "T2", "profile", "distinct sponsoring companies"),
    ("veeva_industry_engagement", "T1", "composite", "mean z of pay_log_total, pay_n_companies, pay_speaker_consult (matched HCPs)"),
    ("veeva_network_breadth", "T1", "composite", "mean z of pub_log_n, event_any, n_society_roles (matched HCPs)"),
]
T3_SPEC = ["veeva_best_decile", "veeva_decile_bin", "veeva_cl_decile_ckd", "veeva_cl_decile_hf", "veeva_cl_decile_t2d",
           "veeva_cl_decile_t1d", "veeva_cl_decile_htn", "veeva_cl_decile_mra_class", "veeva_cl_decile_sglt2_class",
           "veeva_fa_decile_max_inscope", "veeva_dr_any_inscope", "veeva_index_publication", "veeva_index_trial",
           "veeva_index_event", "veeva_index_guideline", "veeva_scientific_reach"]
schema_df = pd.DataFrame(FEATURE_SPEC, columns=["feature", "tier", "family", "rule"]).assign(table="t1_t2")
schema_df = pd.concat([schema_df, pd.DataFrame({"feature": T3_SPEC, "tier": "T3", "family": "influence_rankings", "rule": "snapshot score", "table": "t3"})])
schema_df["position"] = schema_df.groupby("table").cumcount() + 1
spark.createDataFrame(schema_df).write.mode("overwrite").parquet(CFG + "feature_schema_v1")
print(schema_df.groupby(["table", "tier"]).size().to_dict(), "| total:", len(schema_df))

In [0]:
fd = spark.read.parquet(CFG + "focus_dictionary_v1")
# coverage themes
def scored(t, keys, dcol):
    k = fd.filter(f"source_table = '{t}'").select(F.col("focus_name").alias("FOCUS_NAME"), F.col("cl_type").alias("TYPE"), "theme")
    k = k if t == "clinical_leader" else k.drop("TYPE")
    return (rd(t).join(matched, "SE_ID").join(F.broadcast(k), keys)
              .select("SE_ID", "theme", F.col(dcol).cast("int").alias("D")).filter("D BETWEEN 1 AND 10"))
cl_s = scored("clinical_leader", ["FOCUS_NAME", "TYPE"], "SE_DECILE")
fa_s = scored("focus_area", ["FOCUS_NAME"], "FOCUS_DECILE")
dr_s = scored("digital_reach_focus_area", ["FOCUS_NAME"], "SE_DECILE")
themes = cl_s.unionByName(fa_s).unionByName(dr_s).groupBy("SE_ID").agg(F.countDistinct("theme").alias("veeva_n_inscope_themes"))

# T3
TH = {"CKD": "ckd", "Heart failure": "hf", "T2D": "t2d", "T1D": "t1d", "Hypertension": "htn", "Kerendia / MRA": "mra_class", "SGLT2 inhibitors": "sglt2_class"}
cl_p = cl_s.filter(F.col("theme").isin(list(TH))).groupBy("SE_ID").pivot("theme", list(TH)).agg(F.max("D"))
for k, v in TH.items():
    cl_p = cl_p.withColumnRenamed(k, f"veeva_cl_decile_{v}")
IDX = {"Publication Index": "veeva_index_publication", "Clinical Trial Index": "veeva_index_trial",
       "Event Index": "veeva_index_event", "Guideline Index": "veeva_index_guideline", "Scientific Reach": "veeva_scientific_reach"}
idx = rd("index").join(matched, "SE_ID").filter(F.col("TYPE").isin(list(IDX))).groupBy("SE_ID").pivot("TYPE", list(IDX)).agg(F.max(F.col("VALUE").cast("double")))
for k, v in IDX.items():
    idx = idx.withColumnRenamed(k, v)
t3_se = (cl_p.join(fa_s.groupBy("SE_ID").agg(F.max("D").alias("veeva_fa_decile_max_inscope")), "SE_ID", "full")
             .join(dr_s.select("SE_ID").distinct().withColumn("veeva_dr_any_inscope", F.lit(1)), "SE_ID", "full")
             .join(idx, "SE_ID", "full"))

# T2 profile
se = rd("scientific_expert").join(matched, "SE_ID")
deg = F.upper(F.concat_ws("|", "DEGREE_1", "DEGREE_2", "DEGREE_3"))
prof = se.select("SE_ID", F.col("EXPERT_TYPE").alias("veeva_expert_type"),
                 deg.rlike(r"(^|\|)(MD|DO)(\||$)").cast("int").alias("veeva_is_md_do"),
                 deg.rlike(r"(^|\|)PHD(\||$)").cast("int").alias("veeva_is_phd"),
                 (F.col("EMERGING_EXPERT") == "Y").cast("int").alias("veeva_emerging_expert"))
addr = rd("address").join(matched, "SE_ID").select("SE_ID", F.lower(F.concat_ws(" ", "AFFILIATION", "DEPARTMENT")).alias("aff"))
acad = addr.groupBy("SE_ID").agg(F.max(F.col("aff").rlike("universit|college|school of medicine|academic").cast("int")).alias("veeva_is_academic"))
assoc = (rd("association").join(matched, "SE_ID").groupBy("SE_ID").agg(
    F.sum((F.col("ASSOCIATION_TYPE") == "Editorial Board").cast("int")).alias("veeva_n_editorial_boards"),
    F.sum(F.col("ASSOCIATION_TYPE").isin("Scientific Association", "Professional Association").cast("int")).alias("veeva_n_society_roles"),
    F.max(F.lower(F.concat_ws(" ", "ASSOCIATION_NAME", "ASSOCIATION_ORIGINAL_NAME")).rlike("kidney|nephrolog|renal").cast("int")).alias("veeva_kidney_society_flag")))
spons = rd("company_sponsorship").join(matched, "SE_ID").groupBy("SE_ID").agg(F.countDistinct("COMPANY_NAME").alias("veeva_n_sponsors"))
static_se = prof.join(acad, "SE_ID", "left").join(assoc, "SE_ID", "left").join(spons, "SE_ID", "left").join(themes, "SE_ID", "left")
print("profile rows:", prof.count(), "| T3 SE rows:", t3_se.count())

In [0]:
trials, pubs = spark.read.parquet(ACT + "trials"), spark.read.parquet(ACT + "publications")
events, pays = spark.read.parquet(ACT + "events"), spark.read.parquet(ACT + "payments")
CARDIO = ["TAG_CKD", "TAG_CARDIORENAL", "TAG_LANDMARK_TRIAL", "TAG_FINERENONE", "TAG_MRA", "TAG_SGLT2", "TAG_ERA"]

def t1_features(anchor):
    ts, te = window(anchor, "clinical_trial", M_TRIAL_PUB)
    tw = trials.filter((F.col("START_DATE") <= F.lit(te)) & (F.col("END_DATE").isNull() | (F.col("END_DATE") >= F.lit(ts))))
    tr = tw.groupBy("SE_ID").agg(
        F.lit(1).alias("veeva_trial_any_5y"),
        F.log1p(F.countDistinct("CLINICAL_TRIAL_ID")).alias("veeva_trial_log_n_5y"),
        F.max("PI_FLAG").alias("veeva_trial_pi_flag_5y"),
        F.countDistinct(F.when(F.col("TAG_ANY_INSCOPE") == 1, F.col("CLINICAL_TRIAL_ID"))).alias("veeva_trial_n_inscope_5y"),
        F.coalesce(F.max(F.when(F.col("TAG_ANY_INSCOPE") == 1, F.col("PHASE_NUM"))), F.lit(0)).alias("veeva_trial_max_phase_inscope_5y"),
        F.max((F.col("END_DATE").isNull() | (F.col("END_DATE") >= F.lit(te))).cast("int")).alias("veeva_trial_active_flag"),
        F.max("TAG_FINERENONE").alias("veeva_trial_finerenone_flag_5y"),
        F.max(F.greatest("TAG_MRA", "TAG_FINERENONE")).alias("veeva_trial_mra_flag_5y"),
        F.max("TAG_SGLT2").alias("veeva_trial_sglt2_flag_5y"),
        F.countDistinct(F.when(F.greatest(*CARDIO) == 1, F.col("CLINICAL_TRIAL_ID"))).alias("veeva_trial_cardiorenal_n_5y"))

    ps, pe = window(anchor, "publication", M_TRIAL_PUB)
    pw = pubs.filter((F.col("DATE") > F.lit(ps)) & (F.col("DATE") <= F.lit(pe)))
    pb = pw.groupBy("SE_ID").agg(
        F.lit(1).alias("veeva_pub_any_5y"), F.log1p(F.count("*")).alias("veeva_pub_log_n_5y"),
        F.max("SENIOR_AUTHOR_FLAG").alias("veeva_pub_senior_author_flag_5y"),
        F.sum("TAG_ANY_INSCOPE").alias("veeva_pub_n_inscope_5y"))
    last = (pubs.filter(F.col("DATE") <= F.lit(pe)).groupBy("SE_ID")
                .agg(F.floor(F.months_between(F.lit(pe), F.max("DATE"))).cast("int").alias("veeva_pub_months_since_last")))

    es, ee = window(anchor, "medical_event", M_EVENT_PAY)
    ew = events.filter((F.col("START_DATE") > F.lit(es)) & (F.col("START_DATE") <= F.lit(ee)))
    ev = ew.groupBy("SE_ID").agg(
        F.lit(1).alias("veeva_event_any_3y"), F.max("PODIUM_FLAG").alias("veeva_event_podium_flag_3y"),
        F.max("POSTER_FLAG").alias("veeva_event_poster_flag_3y"),
        F.countDistinct(F.when(F.col("TAG_ANY_INSCOPE") == 1, F.col("MEDICAL_EVENT_ID"))).alias("veeva_event_n_inscope_3y"))

    _, ye = window(anchor, "payment", M_EVENT_PAY)
    last_pub_year = ye.year - 1 if ye >= dt.date(ye.year, 6, 30) else ye.year - 2   # program year Y published 30 June Y+1
    yrs = list(range(last_pub_year - 2, last_pub_year + 1))
    pw2 = (pays.filter(F.col("PROGRAM_YEAR").cast("int").isin(yrs))
               .withColumn("AMOUNT", F.expr("try_cast(AMOUNT AS DOUBLE)")))   # AMOUNT is text in the snapshot
    comp = F.lower("COMPANY_NAME").rlike("|".join(COMPETITORS)) & (F.col("BAYER_FLAG") == 0)
    py = pw2.groupBy("SE_ID").agg(
        F.lit(1).alias("veeva_pay_any_3y"), F.log1p(F.sum("AMOUNT")).alias("veeva_pay_log_total_3y"),
        F.log1p(F.sum(F.when(F.col("FOOD_FLAG") == 0, F.col("AMOUNT")).otherwise(F.lit(0.0)))).alias("veeva_pay_excl_food_log_3y"),
        F.countDistinct("COMPANY_NAME").alias("veeva_pay_n_companies_3y"),
        F.max("SPEAKER_CONSULT_FLAG").alias("veeva_pay_speaker_consult_flag_3y"),
        F.max(comp.cast("int")).alias("veeva_pay_competitor_flag_3y"),
        F.max("RESEARCH_FLAG").alias("veeva_pay_research_flag_3y"), F.max("BAYER_FLAG").alias("veeva_pay_bayer_flag_3y"))
    win_log = {"anchor": str(anchor), "trials": f"{ts}..{te}", "publications": f"{ps}..{pe}", "events": f"{es}..{ee}", "payment_program_years": yrs}
    return [tr, pb, last, ev, py], win_log

In [0]:
T1T2_COLS = [f[0] for f in FEATURE_SPEC]
NON_ZERO_FILL = {"veeva_coverage_status", "veeva_has_profile", "veeva_has_inscope_decile", "veeva_expert_type",
                 "veeva_pub_months_since_last", "veeva_industry_engagement", "veeva_network_breadth"}

def zmean(df, cols, name):
    st = df.filter("veeva_has_profile = 1").agg(*[F.mean(c).alias(f"m_{c}") for c in cols], *[F.stddev(c).alias(f"s_{c}") for c in cols]).collect()[0]
    z = [(F.col(c) - st[f"m_{c}"]) / st[f"s_{c}"] for c in cols]
    return df.withColumn(name, sum(z) / len(z))

def build(anchor):
    parts, win_log = t1_features(anchor)
    se_df = matched
    for p in parts + [static_se]:
        se_df = se_df.join(p, "SE_ID", "left")
    hcp = (spine.select("PROVIDER_ID", "SE_ID", "IN_VEEVA", "BEST_DECILE", "VEEVA_COVERAGE_STATUS")
                .join(se_df, "SE_ID", "left")
                .withColumn("veeva_coverage_status", F.col("VEEVA_COVERAGE_STATUS"))
                .withColumn("veeva_has_profile", F.col("IN_VEEVA"))
                .withColumn("veeva_has_inscope_decile", F.col("BEST_DECILE").isNotNull().cast("int")))
    fill0 = [c for c in T1T2_COLS if c not in NON_ZERO_FILL]
    for c in fill0:   # 0 only for HCPs in Veeva; null for NOT_IN_VEEVA (principle 2)
        hcp = hcp.withColumn(c, F.when(F.col("IN_VEEVA") == 1, F.coalesce(F.col(c), F.lit(0))))
    hcp = zmean(hcp, ["veeva_pay_log_total_3y", "veeva_pay_n_companies_3y", "veeva_pay_speaker_consult_flag_3y"], "veeva_industry_engagement")
    hcp = zmean(hcp, ["veeva_pub_log_n_5y", "veeva_event_any_3y", "veeva_n_society_roles"], "veeva_network_breadth")
    t12 = hcp.select("PROVIDER_ID", *T1T2_COLS).withColumn("anchor_month", F.lit(str(anchor))).withColumn("snapshot_id", F.lit(SNAPSHOT_ID))
    t3 = (spine.select("PROVIDER_ID", "SE_ID", "IN_VEEVA", F.col("BEST_DECILE").alias("veeva_best_decile"))
               .join(t3_se, "SE_ID", "left")
               .withColumn("veeva_decile_bin", F.when(F.col("IN_VEEVA") == 0, None).when(F.col("veeva_best_decile").isNull(), "unscored")
                           .when(F.col("veeva_best_decile") >= 8, "8-10").when(F.col("veeva_best_decile") >= 4, "4-7").otherwise("1-3"))
               .withColumn("veeva_dr_any_inscope", F.when(F.col("IN_VEEVA") == 1, F.coalesce("veeva_dr_any_inscope", F.lit(0))))
               .select("PROVIDER_ID", *T3_SPEC).withColumn("anchor_month", F.lit(str(anchor))).withColumn("snapshot_id", F.lit(SNAPSHOT_ID)))
    t12.write.mode("overwrite").parquet(f"{FEAT}t1_t2/anchor={anchor}")
    t3.write.mode("overwrite").parquet(f"{FEAT}t3/anchor={anchor}")
    return win_log

win_logs = [build(a) for a in ANCHORS]
display(pd.DataFrame(win_logs))

In [0]:
exp_t12 = ["PROVIDER_ID"] + T1T2_COLS + ["anchor_month", "snapshot_id"]
exp_t3 = ["PROVIDER_ID"] + T3_SPEC + ["anchor_month", "snapshot_id"]
rows = []
for a in ANCHORS:
    t12 = spark.read.parquet(f"{FEAT}t1_t2/anchor={a}"); t3 = spark.read.parquet(f"{FEAT}t3/anchor={a}")
    leak = t12.filter("veeva_has_profile = 0").select(
        sum(F.col(c).isNotNull().cast("int") for c in T1T2_COLS if c not in NON_ZERO_FILL)).collect()[0][0]
    rows.append({"anchor": str(a), "t1_t2_rows": t12.count(), "t1_t2_distinct_hcp": t12.select("PROVIDER_ID").distinct().count(),
                 "t1_t2_schema_ok": t12.columns == exp_t12, "t3_rows": t3.count(), "t3_schema_ok": t3.columns == exp_t3,
                 "non_null_values_for_not_in_veeva": leak})
chk = pd.DataFrame(rows)
chk["same_columns_both_runs"] = len({tuple(spark.read.parquet(f"{FEAT}t1_t2/anchor={a}").columns) for a in ANCHORS}) == 1
display(chk)
print("A8 pass:", bool(chk.t1_t2_schema_ok.all() and chk.t3_schema_ok.all() and chk.same_columns_both_runs.all()
                      and (chk.t1_t2_rows == 294727).all() and (chk.non_null_values_for_not_in_veeva == 0).all()))

In [0]:
num = [f[0] for f in FEATURE_SPEC if f[0] not in ("veeva_coverage_status", "veeva_expert_type")]
summ = []
for a in ANCHORS:
    d = spark.read.parquet(f"{FEAT}t1_t2/anchor={a}").filter("veeva_has_profile = 1")
    r = d.agg(*[F.mean(c).alias(c) for c in num]).toPandas().T.rename(columns={0: f"mean_{a}"})
    summ.append(r)
summ = pd.concat(summ, axis=1).round(4)
summ["constant_both"] = spark.read.parquet(f"{FEAT}t1_t2/anchor={ANCHORS[0]}").filter("veeva_has_profile = 1").agg(
    *[F.countDistinct(c).alias(c) for c in num]).toPandas().T[0].values <= 1
display(summ.reset_index().rename(columns={"index": "feature"}))